# EarthCARE Phasenprodukte – Ross Sea (Antarktis)

Dieses Notebook lädt EarthCARE-Phasenprodukte für das Ross Sea-Gebiet in der Antarktis herunter.

**Phasenprodukte** beschreiben den thermodynamischen Zustand (flüssig / Eis / gemischt) von Hydrometeoren:

| Produkt | Instrument | Interner Name | Beschreibung |
|---|---|---|---|
| `CPR_TC__2A` | CPR (Radar)   | C-TC  | Target Classification – Wolkenphase aus Radar |
| `ATL_TC__2A` | ATLID (Lidar) | A-TC  | Target Classification – Wolken- und Aerosoltypen |
| `ATL_ICE_2A` | ATLID (Lidar) | A-ICE | Eis-Mikrophysik (Effektivradius, IWC) |
| `CPR_CD__2A` | CPR (Radar)   | C-CD  | Cloud Detection – Radarbasierte Wolkenerkennung |
| `MSI_CM__2A` | MSI (Imager)  | **M-CP** | Cloud Phase & Cloud Mask – thermodynamische Phase aus Spektralkanälen (0,67 µm / 1,65 µm / IR) |
| `MSI_COP_2A` | MSI (Imager)  | M-COP | Cloud Optical Properties – baut auf M-CP auf (LWP, IWP, Reff) |

> **Hinweis zu M-CP:** Das MSI Cloud Phase-Produkt ist keine eigenständige Datei, sondern eine Variable (`cloud_phase`) innerhalb von `MSI_CM__2A`. Es unterscheidet: *clear / liquid / supercooled / ice / mixed / overlap*.

**Untersuchungsgebiet: Ross Sea**
- Breitengrad: 70°S – 85°S
- Längengrad: 160°E – 210°E (≡ 160°E – 150°W)

**Datenquelle:** ESA MAAP Catalogue via [`earthcare-downloader`](https://github.com/actris-cloudnet/earthcare-downloader)  
**Authentifizierung:** MAAP Offline Token (siehe Abschnitt 3)

## 1  Installation

In [ ]:
%pip install earthcare-downloader netCDF4 h5py xarray matplotlib cartopy --quiet

## 2  Imports

In [ ]:
import os
import asyncio
from datetime import date, timedelta
from pathlib import Path

import h5py
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from earthcare_downloader.aio import search, download

## 3  Authentifizierung

Ein **MAAP Offline Token** ist erforderlich. Token beziehen:

1. Öffne: https://portal.maap.eo.esa.int/ini/services/auth/token/
2. Melde dich mit ESA EO-Sign-In an
3. Kopiere den Token-String und setze ihn unten ein

Der Token ist 90 Tage gültig. Alternativ kann er in der Umgebungsvariable `MAAP_TOKEN` gesetzt werden.

In [ ]:
# Token aus Umgebungsvariable lesen oder hier direkt eintragen
MAAP_TOKEN = os.environ.get("MAAP_TOKEN", "DEIN_TOKEN_HIER")

if MAAP_TOKEN == "DEIN_TOKEN_HIER":
    raise ValueError(
        "Bitte MAAP_TOKEN setzen:\n"
        "  export MAAP_TOKEN='<dein-token>'  (Shell)\n"
        "oder direkt in dieser Zelle eintragen."
    )

# Token als Umgebungsvariable setzen (wird von earthcare-downloader automatisch erkannt)
os.environ["MAAP_TOKEN"] = MAAP_TOKEN
print("Token gesetzt ✓")

## 4  Konfiguration

Zeitraum, Ausgabeverzeichnis und Phasenprodukte können hier angepasst werden.

In [ ]:
# --- Zeitraum ---
START_DATE = date(2025, 1, 1)   # Anfang (YYYY-MM-DD)
END_DATE   = date(2025, 1, 31)  # Ende    (YYYY-MM-DD)

# --- Ross Sea Bounding Box ---
# Das Ross Sea liegt grob zwischen 160°E und 150°W, 70°S bis 85°S.
# Da es die Datumsgrenze überschreitet, werden zwei Längengradbereiche genutzt.
LAT_MIN = -85.0
LAT_MAX = -70.0
LON_MIN_EAST  = 160.0   # östlicher Teil:  160°E – 180°
LON_MAX_EAST  = 180.0
LON_MIN_WEST  = -180.0  # westlicher Teil: 180° – 150°W
LON_MAX_WEST  = -150.0

# --- Phasenprodukte ---
PHASE_PRODUCTS = [
    "CPR_TC__2A",  # CPR Target Classification (Radar-Wolkenphase)
    "ATL_TC__2A",  # ATLID Target Classification (Lidar-Wolken- & Aerosoltyp)
    "ATL_ICE_2A",  # ATLID Eis-Mikrophysik
    "CPR_CD__2A",  # CPR Cloud Detection
    "MSI_CM__2A",  # MSI Cloud Mask – enthält M-CP (cloud_phase Variable)
    "MSI_COP_2A",  # MSI Cloud Optical Properties (LWP, IWP, Reff)
]

# --- Ausgabeverzeichnis ---
OUT_DIR = Path("data/ross_sea")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Zeitraum  : {START_DATE} – {END_DATE}")
print(f"Gebiet    : {LAT_MIN}°N – {LAT_MAX}°N, {LON_MIN_EAST}°E – {LON_MAX_WEST}°W")
print(f"Produkte  : {', '.join(PHASE_PRODUCTS)}")
print(f"Ausgabe   : {OUT_DIR.resolve()}")

## 5  Daten suchen

Die Suche fragt den ESA MAAP STAC-Katalog ab und gibt verfügbare Granule zurück.
Da das Ross Sea die Datumsgrenze überschreitet, werden zwei Suchanfragen pro Produkt gestellt und kombiniert.

In [ ]:
async def search_ross_sea(product: str, start: date, end: date) -> list:
    """Sucht Granule in beiden Längengradintervallen des Ross Sea."""
    # Östlicher Teil (160°E – 180°)
    results_east = await search(
        product=product,
        start=start.isoformat(),
        stop=end.isoformat(),
        lat_min=LAT_MIN,
        lat_max=LAT_MAX,
        lon_min=LON_MIN_EAST,
        lon_max=LON_MAX_EAST,
    )
    # Westlicher Teil (180° – 150°W)
    results_west = await search(
        product=product,
        start=start.isoformat(),
        stop=end.isoformat(),
        lat_min=LAT_MIN,
        lat_max=LAT_MAX,
        lon_min=LON_MIN_WEST,
        lon_max=LON_MAX_WEST,
    )
    # Duplikate entfernen (gleiche Datei kann in beiden Bereichen auftauchen)
    seen = set()
    combined = []
    for item in results_east + results_west:
        key = getattr(item, "name", str(item))
        if key not in seen:
            seen.add(key)
            combined.append(item)
    return combined


# Alle Produkte suchen
all_results: dict[str, list] = {}

for product in PHASE_PRODUCTS:
    print(f"Suche {product} ...", end=" ")
    found = await search_ross_sea(product, START_DATE, END_DATE)
    all_results[product] = found
    print(f"{len(found)} Granule gefunden")

total = sum(len(v) for v in all_results.values())
print(f"\nGesamt: {total} Granule")

### 5.1  Übersicht der Suchergebnisse

In [ ]:
print(f"{'Produkt':<15} {'Granule':>8}")
print("-" * 25)
for product, items in all_results.items():
    print(f"{product:<15} {len(items):>8}")
print("-" * 25)
print(f"{'Gesamt':<15} {total:>8}")

## 6  Daten herunterladen

Die Granule werden in Unterordner je Produkttyp gespeichert. Bereits vorhandene Dateien werden übersprungen.

In [ ]:
downloaded_paths: dict[str, list[Path]] = {}

for product, items in all_results.items():
    if not items:
        print(f"{product}: Keine Granule – übersprungen.")
        continue

    product_dir = OUT_DIR / product
    product_dir.mkdir(exist_ok=True)

    print(f"Lade {len(items)} Granule für {product} ...")
    paths = await download(
        items,
        output_dir=product_dir,
        max_workers=4,
    )
    downloaded_paths[product] = list(paths)
    total_mb = sum(p.stat().st_size for p in paths if p.exists()) / 1e6
    print(f"  → {len(paths)} Dateien, {total_mb:.1f} MB gespeichert in {product_dir}")

print("\nDownload abgeschlossen.")

### 6.1  Download-Zusammenfassung

In [ ]:
print(f"{'Produkt':<15} {'Dateien':>8} {'Größe (MB)':>12}")
print("-" * 37)
grand_total_files = 0
grand_total_mb = 0.0
for product, paths in downloaded_paths.items():
    n = len(paths)
    mb = sum(p.stat().st_size for p in paths if p.exists()) / 1e6
    grand_total_files += n
    grand_total_mb += mb
    print(f"{product:<15} {n:>8} {mb:>12.1f}")
print("-" * 37)
print(f"{'Gesamt':<15} {grand_total_files:>8} {grand_total_mb:>12.1f}")

## 7  Dateistruktur inspizieren

EarthCARE Level-2-Produkte liegen als HDF5-Dateien vor. Hier wird die Struktur eines Beispielgranuls ausgegeben.

In [ ]:
def print_hdf5_tree(name, obj, indent=0):
    prefix = "  " * indent
    if isinstance(obj, h5py.Dataset):
        print(f"{prefix}📄 {name.split('/')[-1]}  shape={obj.shape}  dtype={obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"{prefix}📁 {name.split('/')[-1]}/")


# Erstes verfügbares Granul anzeigen
example_file = None
example_product = None

for product, paths in downloaded_paths.items():
    if paths:
        example_file = paths[0]
        example_product = product
        break

if example_file and example_file.exists():
    print(f"Datei: {example_file.name}  ({example_file.stat().st_size / 1e6:.1f} MB)")
    print(f"Produkt: {example_product}\n")
    with h5py.File(example_file, "r") as f:
        f.visititems(lambda name, obj: print_hdf5_tree(name, obj, indent=name.count("/")))
else:
    print("Keine Dateien vorhanden – Struktur kann nicht angezeigt werden.")

## 8  Schnellansicht: CPR Target Classification

Das Produkt `CPR_TC__2A` enthält eine Zielklassifikation, die u.a. Wolkeneis, flüssiges Wasser und Niederschlag unterscheidet. Hier wird das erste heruntergeladene Granul visualisiert.

In [ ]:
CPR_TC_CLASSES = {
    0:  ("Klar",                     "#ffffff"),
    1:  ("Flüssigwasser-Wolke",       "#3399ff"),
    2:  ("Eiswolke",                  "#99ccff"),
    3:  ("Mischphasen-Wolke",         "#9933ff"),
    4:  ("Regen",                     "#0033cc"),
    5:  ("Schnee/Graupel",            "#66ffff"),
    6:  ("Niederschlag (gemischt)",   "#330099"),
    7:  ("Aerosol",                   "#ffcc00"),
    8:  ("Insekt / Artefakt",         "#ff6600"),
    -1: ("Unbekannt/Ungültig",        "#cccccc"),
}


def plot_cpr_tc(filepath: Path):
    with h5py.File(filepath, "r") as f:
        # Typische Pfade – ggf. an tatsächliche HDF5-Struktur anpassen
        # Suche nach dem Target-Classification-Datensatz
        tc_key = None
        lat_key = None

        def find_keys(name, obj):
            nonlocal tc_key, lat_key
            lname = name.lower()
            if isinstance(obj, h5py.Dataset):
                if "target_classif" in lname or "tc" in lname.split("/")[-1]:
                    tc_key = name
                if "latitude" in lname or "lat" == lname.split("/")[-1]:
                    lat_key = name

        f.visititems(find_keys)

        if tc_key is None:
            print("Target-Classification-Variable nicht gefunden.")
            return

        tc   = f[tc_key][()]      # shape: (along_track, range_bins)
        lats = f[lat_key][()] if lat_key else np.arange(tc.shape[0])

    # Farbskala
    classes = sorted(CPR_TC_CLASSES.keys())
    colors  = [CPR_TC_CLASSES[c][1] for c in classes]
    cmap    = mcolors.ListedColormap(colors)
    bounds  = [c - 0.5 for c in classes] + [classes[-1] + 0.5]
    norm    = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(14, 4))
    img = ax.pcolormesh(
        lats, np.arange(tc.shape[1]), tc.T,
        cmap=cmap, norm=norm, shading="auto",
    )
    cbar = fig.colorbar(img, ax=ax, ticks=classes, pad=0.01)
    cbar.set_ticklabels([CPR_TC_CLASSES[c][0] for c in classes], fontsize=8)
    ax.set_xlabel("Breitengrad (°)")
    ax.set_ylabel("Höhenbin")
    ax.set_title(f"CPR Target Classification – {filepath.name}")
    plt.tight_layout()
    plt.show()


cpr_files = downloaded_paths.get("CPR_TC__2A", [])
if cpr_files:
    plot_cpr_tc(cpr_files[0])
else:
    print("Keine CPR_TC__2A-Dateien vorhanden.")

## 8b  Schnellansicht: MSI Cloud Phase (M-CP)

Die Variable `cloud_phase` in `MSI_CM__2A` gibt die thermodynamische Phase pixelweise für den gesamten MSI-Swath an (150 km breit). Klassen: *clear · liquid · supercooled · ice · mixed · overlap*.

In [ ]:
MSI_PHASE_CLASSES = {
    0: ("Clear",       "#ffffff"),
    1: ("Liquid",      "#3399ff"),
    2: ("Supercooled", "#00ccff"),
    3: ("Ice",         "#99ccff"),
    4: ("Mixed",       "#9933ff"),
    5: ("Overlap",     "#ff9900"),
    6: ("Ungültig",    "#cccccc"),
}


def plot_msi_cloud_phase(filepath: Path):
    with h5py.File(filepath, "r") as f:
        phase_key = None
        lat_key   = None

        def find_keys(name, obj):
            nonlocal phase_key, lat_key
            if not isinstance(obj, h5py.Dataset):
                return
            lname = name.lower()
            if "cloud_phase" in lname or "phase" in lname.split("/")[-1]:
                phase_key = name
            if "latitude" in lname or lname.split("/")[-1] == "lat":
                lat_key = name

        f.visititems(find_keys)

        if phase_key is None:
            print("cloud_phase-Variable nicht in dieser Datei gefunden.")
            return

        phase = f[phase_key][()]
        lats  = f[lat_key][()] if lat_key else np.arange(phase.shape[0])

    # Bei 2-D MSI-Daten (along_track × across_track) Mittelwert der Querbahn-Lats nehmen
    if lats.ndim == 2:
        lats = lats[:, lats.shape[1] // 2]

    classes = sorted(MSI_PHASE_CLASSES.keys())
    colors  = [MSI_PHASE_CLASSES[c][1] for c in classes]
    labels  = [MSI_PHASE_CLASSES[c][0] for c in classes]
    cmap    = mcolors.ListedColormap(colors)
    bounds  = [c - 0.5 for c in classes] + [classes[-1] + 0.5]
    norm    = mcolors.BoundaryNorm(bounds, cmap.N)

    # Phase kann 1-D (along_track,) oder 2-D (along_track, across_track) sein
    data = phase if phase.ndim == 2 else phase[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(14, 4))
    img = ax.pcolormesh(
        lats, np.arange(data.shape[1]), data.T,
        cmap=cmap, norm=norm, shading="auto",
    )
    cbar = fig.colorbar(img, ax=ax, ticks=classes, pad=0.01)
    cbar.set_ticklabels(labels, fontsize=8)
    ax.set_xlabel("Breitengrad (°)")
    ax.set_ylabel("Across-Track-Pixel")
    ax.set_title(f"MSI Cloud Phase (M-CP) – {filepath.name}")
    plt.tight_layout()
    plt.show()


msi_cm_files = downloaded_paths.get("MSI_CM__2A", [])
if msi_cm_files:
    plot_msi_cloud_phase(msi_cm_files[0])
else:
    print("Keine MSI_CM__2A-Dateien vorhanden.")

## 9  Überflugpfade auf Karte darstellen

In [ ]:
def load_track_coordinates(filepath: Path) -> tuple[np.ndarray, np.ndarray] | None:
    """Liest Längen- und Breitengrad aus einem EarthCARE HDF5-Granul."""
    with h5py.File(filepath, "r") as f:
        lat, lon = None, None

        def find_coords(name, obj):
            nonlocal lat, lon
            if not isinstance(obj, h5py.Dataset):
                return
            lname = name.lower().split("/")[-1]
            if lname in ("latitude", "lat") and lat is None:
                lat = obj[()].ravel()
            if lname in ("longitude", "lon") and lon is None:
                lon = obj[()].ravel()

        f.visititems(find_coords)
        return (lat, lon) if lat is not None and lon is not None else None


fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.SouthPolarStereo())
ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#d4b896")
ax.add_feature(cfeature.OCEAN, facecolor="#c8e0f0")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.gridlines(draw_labels=False, linewidth=0.4, linestyle="--", color="gray")

import matplotlib.patches as mpatches

# Ross Sea-Box (vereinfacht, östlich der Datumsgrenze)
for lon_box in [(160, 180), (-180, -150)]:
    lons = [lon_box[0], lon_box[1], lon_box[1], lon_box[0], lon_box[0]]
    lats = [LAT_MAX, LAT_MAX, LAT_MIN, LAT_MIN, LAT_MAX]
    ax.plot(lons, lats, transform=ccrs.PlateCarree(),
            color="red", linewidth=1.5, linestyle="--", label="Ross Sea")

colors_map = {
    "CPR_TC__2A": "blue",
    "ATL_TC__2A": "orange",
    "ATL_ICE_2A": "green",
    "CPR_CD__2A": "purple",
    "MSI_CM__2A": "red",
    "MSI_COP_2A": "brown",
}
plotted_products = set()

for product, paths in downloaded_paths.items():
    color = colors_map.get(product, "black")
    for fp in paths[:10]:  # max. 10 Überflüge pro Produkt anzeigen
        coords = load_track_coordinates(fp)
        if coords is None:
            continue
        lat_t, lon_t = coords
        label = product if product not in plotted_products else None
        ax.plot(lon_t, lat_t, transform=ccrs.PlateCarree(),
                color=color, linewidth=0.8, alpha=0.7, label=label)
        plotted_products.add(product)

ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"EarthCARE Überflugpfade – Ross Sea\n{START_DATE} – {END_DATE}")
plt.tight_layout()
plt.show()

## 10  Nächste Schritte

- **Zeitraum anpassen**: `START_DATE` und `END_DATE` in Zelle 4 ändern  
- **Weitere Produkte**: z.B. `ATL_FM__2A` (Feature Mask) oder `CPR_CLD_2A` (Cloud Detection) ergänzen  
- **Synergieprodukte**: L2B-Produkte wie `AC__TC_2B` (ATL+CPR Target Classification) für kombinierte Phase-Analyse  
- **Verarbeitung**: Dateien mit `xarray` und `h5py` einlesen, Statistiken über Eiswolken-Häufigkeit berechnen